In [1]:
import torch
import src.ADG as adg
import random

# Modèle + tokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
model, tokenizer = adg.load_model("gpt2", device)

c:\Users\steph_anaconda\anaconda3\envs\lab_ML\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2953.22it/s]

Loaded gpt2 on cpu (vocab size: 50257)


In [2]:
# Configuration ADG
config = adg.ADGConfig(model=model, tokenizer=tokenizer,
                       device=device, temperature=1, top_k=50)

In [3]:
SEED = 1976
random.seed(SEED)
torch.manual_seed(SEED)


prompt = "Little white rabbit, are you really going to"
prompt_ids = adg.encode_prompt(prompt, config)

print(f"PROMPT : {prompt}")

# Bits à cacher + prompt de couverture
bits = [1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0]

# Cover : Alice genère une séquence sans message caché  
cover = adg.ADG_generate_cover(prompt_ids, nb_bits=len(bits),
                               max_tokens=100, config=config )
print(f"COVER  : {cover.text}")


#Stego : Alice genère une séquence dissimulant  le payload
stego = adg.ADG_encode(prompt_ids, bits, config)
print(f"STEGO  : {stego.text}  \n")

print("embedding rate pour le stego :", stego.embedding_rate, "bits/token")

PROMPT : Little white rabbit, are you really going to
COVER  :  take this one? This isn
STEGO  :  put them inside your car  

embedding rate pour le stego : 4.0 bits/token


In [4]:
# Bob reconstitue le payload, il rejoue le routage pour retrouver les bits.
recovered = adg.ADG_decode(prompt_ids, stego.tokens, len(bits), config)
print("payload reconstruit :", recovered)
print("roundtrip ok:", recovered == bits)   

payload reconstruit : [1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0]
roundtrip ok: True
